In [1]:
import sys
from pathlib import Path

PIPELINE_DIR = Path.cwd().parent
if str(PIPELINE_DIR) not in sys.path:
    sys.path.insert(0, str(PIPELINE_DIR))

import config
from rx import io, text, places, remote
print(config.PROJECT_DIR)

/Users/larawashington/Documents/Parsons/MajorStudio1/Projects/revolutionary-xroads


In [2]:
con = remote.connect()
PENSION = remote.hf_url("pension")
print(PENSION)

con.sql(f"""
        select count(*) as pages,
        count(distinct NAID) as files
        from '{PENSION}'
""").df()

hf://datasets/RevolutionCrossroads/nara_revolutionary_war_pension_files@90f90867333c320af57a2ed1f43d5187c6685893/data/train-*.parquet


,pages,files
0,2195663,78745


In [3]:
files_raw = con.sql(f"""
                    select NAID, any_value(title) as title, count(*) as n_pages
                    from '{PENSION}'
                    group by NAID
""").df()
io.write_stage(files_raw, "interim", "files_raw")
files_raw.shape

(78745, 3)

In [4]:
pages_index = con.sql(f"""
    select NAID, pageNumber,
           coalesce(trim(transcriptionText) <> '', false) as has_transcription,
           coalesce(trim(RevX_OCR_text)     <> '', false) as has_revx_ocr,
           coalesce(trim(fs_extractedText)  <> '', false) as has_fs_text
    from '{PENSION}'
""").df()
io.write_stage(pages_index, "interim", "pages_index")
pages_index.shape


(2195663, 5)

Find every declaration page using the oath structure: 
"(his|her) oath make the following declaration" = 1818/1832/widow acts
"on his oath declare(s)" = 1820 schedules
"declaration in order to obtain"

In [5]:
TXT = "lower(coalesce(nullif(trim(transcriptionText), ''), nullif(trim(RevX_OCR_text), ''), fs_extractedText))"
DECL_RX = r"(his|her) oath,? (make the following declaration|declares?\b)|declaration in order to obtain"

decl_keys = con.sql(f"""
    with p as (
        select NAID, pageNumber,
               regexp_matches({TXT}, '{DECL_RX}') as is_decl,
               {TXT} like '%personally appeared%' as is_pa
        from '{PENSION}'
    ),
    d as (select NAID, pageNumber, 'declaration' as kind from p where is_decl),
    f as (select NAID, pageNumber, 'fallback' as kind from p
          where is_pa and NAID not in (select NAID from d)
          qualify row_number() over (partition by NAID order by pageNumber) = 1)
    select * from d union all select * from f
""").df()
io.write_stage(decl_keys, "interim", "decl_keys")
decl_keys.groupby("kind").agg(pages=("NAID", "size"), files=("NAID", "nunique"))


,pages,files
kind,,
declaration,110161,56748
fallback,6282,6282


In [6]:
want = con.sql("""
    select NAID, pageNumber + k as pageNumber, min(k) as offs, arg_min(kind, k) as kind
    from decl_keys, range(0, 3) t(k)
    group by 1, 2
""").df()
print(want.offs.value_counts().sort_index())

SRC = """case when nullif(trim(transcriptionText), '') is not null then 'transcriptionText'
              when nullif(trim(RevX_OCR_text), '')     is not null then 'RevX_OCR_text'
              when nullif(trim(fs_extractedText), '')  is not null then 'fs_extractedText' end"""
RAW = "coalesce(nullif(trim(transcriptionText), ''), nullif(trim(RevX_OCR_text), ''), fs_extractedText)"
pages_decl = con.sql(f"""
    select p.NAID, p.pageNumber, w.kind, w.offs, {SRC} as text_source, {RAW} as text
    from '{PENSION}' p join want w using (NAID, pageNumber)
""").df()
io.write_stage(pages_decl, "interim", "pages_decl")
pages_decl.shape, pages_decl.NAID.nunique()

offs
0    116443
1    113896
2    108760
Name: count, dtype: int64


((334409, 6), 63030)

In [ ]:
assert files_raw.NAID.is_unique and len(files_raw) == 78745
assert files_raw.n_pages.sum() == len(pages_index) == 2195663
assert not pages_decl.duplicated(["NAID", "pageNumber"]).any()      # one row per page, not per file
assert pages_decl.NAID.isin(files_raw.NAID).all()
assert pages_decl.offs.between(0, 2).all()
print("A01 ok")

A01 ok


reason_pages = sworn reasons for removing. pensioner who moved states had to transfer his pension to a new agency. 

In [8]:
REASON_RX = r"reasons? (?:for|of) (?:his |her |my )?(?:removing|removal|having removed|moving|leaving)|cause of (?:his |her )?remov"
reason_pages = con.sql(f"""
    select NAID, pageNumber, {TXT} as text
    from '{PENSION}'
    where regexp_matches({TXT}, '{REASON_RX}')
""").df()
io.write_stage(reason_pages, "interim", "reason_pages")
reason_pages.shape, reason_pages.NAID.nunique()

((2493, 3), 2079)